# Part 5 | Session 34: Claude Code를 이용한 AI Agent 구현 실습

**© Copyright AIDENTIFY. All rights reserved.**

## 0️⃣ 환경 준비 — VSCode에서 Claude Code로 바이브코딩

이 실습은 **VSCode + Claude Code** 로 *바이브코딩*(자연어로 코드를 만들고 고치는 방식) 환경을 준비하는 것부터 시작합니다.
**AWS 리눅스 서버에 SSH로 접속**해 작업하는 것을 전제로 합니다.

> **키가 2개인 이유 — 역할이 다릅니다.**
> - **Claude (Anthropic)** → *바이브코딩 도구*(Claude Code) 전용. VSCode에서 코드를 만들고 고치는 데 사용.
> - **OpenAI** → *노트북 안 LLM 실습*(Agent / Tool Use)용. 아래 코드가 호출하는 모델은 **OpenAI** 입니다.

### 1. VSCode에서 원격(AWS)에 접속
- 확장 **Remote - SSH** 설치
- `Ctrl+Shift+P` → **Remote-SSH: Connect to Host** → AWS 인스턴스 선택
- 터미널 프롬프트가 `ubuntu@...` 로 뜨면 원격 접속 성공

### 2. Claude Code 확장 설치 (바이브코딩용, 반드시 원격 쪽)
- Extensions(`Ctrl+Shift+X`)에서 **Claude Code** 검색 → 설치
- ⚠️ 설치 버튼이 **"Install in SSH: (서버명)"** 으로 보여야 합니다. 로컬에만 깔면 서버의 키를 못 읽습니다.
- Claude Code 로그인용 키 등록(터미널):
  ```bash
  echo 'export ANTHROPIC_API_KEY=sk-ant-***' >> ~/.bashrc
  source ~/.bashrc
  ```

### 3. 노트북 LLM 실습용 OpenAI 키 — `.env` 파일 사용
이 노트북의 Agent 코드는 **OpenAI** 모델을 호출합니다. 키는 코드에 직접 쓰지 않고 **노트북과 같은 폴더의 `.env` 파일**에 둡니다.

`.env` 파일 내용:
```
OPENAI_API_KEY=sk-proj-***
```
- `.env` 는 **git에 커밋하지 마세요** (`.gitignore` 에 `.env` 추가)
- 아래 코드가 `python-dotenv` 로 이 파일을 읽어 `OPENAI_API_KEY` 를 불러옵니다. (미설치 시 `pip install python-dotenv`)

### 4. VSCode에 반영
- `Ctrl+Shift+P` → **Developer: Reload Window**

> 준비가 끝났으면 아래 셀을 실행해 환경이 제대로 잡혔는지 확인하세요.

In [ ]:
# 환경 준비 확인 — .env 의 OpenAI 키(LLM 실습용) & Claude Code(바이브코딩) 점검
import os, shutil

# python-dotenv 로 같은 폴더의 .env 로드
try:
    from dotenv import load_dotenv
    load_dotenv()
except ModuleNotFoundError:
    print("ℹ️ python-dotenv 미설치 — 설치: pip install python-dotenv")

# 1) 노트북 LLM 실습용 OpenAI 키 (.env 에서 로드, 키 값 자체는 출력하지 않음)
openai_key = os.environ.get("OPENAI_API_KEY")
if openai_key and openai_key.startswith("sk-"):
    print(f"✅ OPENAI_API_KEY 로드됨 (sk-...{openai_key[-4:]}) — 노트북 LLM 실습 준비 완료")
else:
    print("❌ OPENAI_API_KEY 없음 — 노트북과 같은 폴더에 .env 파일을 만들고 아래 한 줄을 넣으세요:")
    print("   OPENAI_API_KEY=sk-proj-***")

# 2) 바이브코딩 도구 Claude Code (CLI 설치 여부만 확인 — 노트북 LLM 실행에는 불필요)
claude_path = shutil.which("claude")
if claude_path:
    print(f"✅ Claude Code CLI 설치됨 — {claude_path} (바이브코딩용)")
else:
    print("ℹ️ Claude Code CLI 미설치 — VSCode 확장으로 바이브코딩 가능 / CLI 필요시: npm install -g @anthropic-ai/claude-code")

## 1️⃣ Claude Code 소개

**Claude Code**는 Anthropic에서 개발한 **CLI 기반 AI 코딩 어시스턴트**입니다.
터미널에서 직접 실행하여 코드 생성, 파일 편집, 프로젝트 관리 등을 자연어로 수행할 수 있습니다.

### 주요 특징
- **Agentic 코딩**: 파일 읽기/쓰기, 명령어 실행 등을 자율적으로 수행
- **프로젝트 컨텍스트 이해**: 전체 코드베이스를 파악하여 일관된 코드 생성
- **터미널 통합**: 별도 IDE 없이 터미널에서 바로 사용
- **Git 연동**: 커밋, PR 생성 등 Git 워크플로우 지원

### 설치 및 기본 사용법

```bash
# Claude Code 설치 (npm 사용)
npm install -g @anthropic-ai/claude-code

# 프로젝트 디렉토리에서 실행
cd my-project
claude

# 한 줄 명령으로 실행
claude "이 프로젝트의 구조를 설명해줘"
claude "테스트 코드를 작성해줘"
```

### 주요 명령어

| 명령어 | 설명 |
|--------|------|
| `claude` | 대화형 모드 시작 |
| `claude "prompt"` | 단일 프롬프트 실행 |
| `claude -p "prompt"` | 파이프 모드 (스크립트 연동) |
| `/help` | 도움말 보기 |
| `/clear` | 대화 초기화 |
| `/cost` | 현재 세션 비용 확인 |

## 2️⃣ Claude Code로 프로젝트 생성

Claude Code를 사용하면 자연어 지시만으로 전체 프로젝트를 생성할 수 있습니다.

### 예시: 간단한 FastAPI 프로젝트 생성

```
$ claude

> FastAPI로 할 일(TODO) 관리 REST API를 만들어줘.
> - CRUD 엔드포인트 (생성, 조회, 수정, 삭제)
> - SQLite 데이터베이스 사용
> - Pydantic 모델로 요청/응답 스키마 정의
> - 기본 에러 핸들링 포함
```

Claude Code가 자동으로 수행하는 작업:
1. 프로젝트 디렉토리 구조 생성
2. `requirements.txt` 작성
3. 데이터베이스 모델 정의
4. API 엔드포인트 구현
5. 필요시 테스트 코드 작성

### Claude Code가 생성하는 프로젝트 구조 예시

```
todo-api/
├── main.py              # FastAPI 앱 & 엔드포인트
├── models.py            # SQLAlchemy 모델
├── schemas.py           # Pydantic 스키마
├── database.py          # DB 연결 설정
├── requirements.txt     # 의존성 패키지
└── tests/
    └── test_main.py     # API 테스트
```

> **Point**: Claude Code는 단순히 코드를 생성하는 것이 아니라,
> 파일 시스템을 직접 조작하여 실제 프로젝트를 만들어냅니다.

## 3️⃣ AI Agent 개념

AI Agent는 단순히 텍스트를 생성하는 LLM을 넘어, **자율적으로 행동하고 환경과 상호작용**할 수 있는 시스템입니다.

### Agent의 4가지 핵심 구성 요소

```
┌─────────────────────────────────┐
│           AI Agent              │
│                                 │
│  ┌─────────┐  ┌──────────┐     │
│  │   LLM   │  │ Planning │     │
│  │ (두뇌)   │  │ (계획)    │     │
│  └─────────┘  └──────────┘     │
│                                 │
│  ┌─────────┐  ┌──────────┐     │
│  │  Tools  │  │  Memory  │     │
│  │ (도구)   │  │ (기억)    │     │
│  └─────────┘  └──────────┘     │
│                                 │
└─────────────────────────────────┘
```

| 구성 요소 | 역할 | 예시 |
|-----------|------|------|
| **LLM** | 추론 및 의사결정 | Claude, GPT-4 등 |
| **Tools** | 외부 시스템과 상호작용 | API 호출, DB 조회, 웹 검색 |
| **Memory** | 이전 상호작용 기억 | 대화 히스토리, 벡터 DB |
| **Planning** | 작업 분해 및 실행 계획 | 단계별 계획 수립, 자기 성찰 |

### Agent vs. 단순 LLM 호출

| 특성 | 단순 LLM 호출 | AI Agent |
|------|---------------|----------|
| 상호작용 | 1회 질문-응답 | 다단계 반복 실행 |
| 도구 사용 | 불가 | 외부 도구 호출 가능 |
| 자율성 | 없음 | 스스로 계획하고 실행 |
| 환경 인식 | 제한적 | 실시간 정보 접근 |
| 오류 처리 | 사용자가 재시도 | 자동 재시도 및 대안 탐색 |

## 4️⃣ Function Calling / Tool Use 개념 복습

Agent의 핵심은 **Tool Use (Function Calling)** 입니다.
LLM이 직접 행동하는 것이 아니라, **어떤 도구를 어떤 인자로 호출할지 결정**하고,
실행 결과를 받아 다음 행동을 결정합니다.

### Tool Use 흐름

```
사용자: "서울 날씨 알려줘"
    ↓
LLM: tool_use(name="get_weather", input={"city": "Seoul"})
    ↓
시스템: 실제 API 호출 → {"temp": 18, "condition": "맑음"}
    ↓
LLM: "서울의 현재 날씨는 18°C이며 맑습니다."
```

In [ ]:
# Tool(함수) 정의 예시
# OpenAI Function Calling 에서 사용하는 tools 파라미터 형식

tools_definition = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "지정된 도시의 현재 날씨 정보를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "날씨를 조회할 도시 이름 (예: Seoul, Busan)"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "온도 단위 (기본값: celsius)"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

print("Tool 정의 완료:")
for tool in tools_definition:
    fn = tool["function"]
    print(f"  - {fn['name']}: {fn['description']}")

## 5️⃣ OpenAI API를 활용한 간단한 Agent 구현

이제 OpenAI의 **Function Calling**(`tools` 파라미터)을 사용하여 간단한 AI Agent를 구현해 보겠습니다.

> 역할 분리: **바이브코딩은 Claude Code**, **노트북 안 LLM 호출은 OpenAI** 입니다.

### 구현할 Agent 기능
- 사용자가 자연어로 질문
- Agent가 필요한 도구를 선택하여 호출
- 도구 실행 결과를 바탕으로 최종 응답 생성

In [ ]:
import os
import json
from dotenv import load_dotenv

# .env 에서 OPENAI_API_KEY 로드 (0️⃣ 환경 준비 참고 — 키를 코드에 직접 쓰지 않음)
load_dotenv()

# 시뮬레이션용 도구 함수들
def get_weather(city: str, unit: str = "celsius") -> dict:
    """날씨 정보를 반환하는 시뮬레이션 함수"""
    weather_data = {
        "Seoul": {"temp": 18, "condition": "맑음", "humidity": 45},
        "Busan": {"temp": 21, "condition": "구름 조금", "humidity": 60},
        "Jeju": {"temp": 22, "condition": "흐림", "humidity": 70},
    }
    data = weather_data.get(city, {"temp": 15, "condition": "정보 없음", "humidity": 50})
    if unit == "fahrenheit":
        data["temp"] = round(data["temp"] * 9/5 + 32, 1)
    data["city"] = city
    data["unit"] = unit
    return data


def search_restaurant(city: str, cuisine: str = "한식") -> dict:
    """레스토랑 검색 시뮬레이션 함수"""
    restaurants = {
        ("Seoul", "한식"): [{"name": "광화문국밥", "rating": 4.5}, {"name": "종로냉면", "rating": 4.3}],
        ("Seoul", "일식"): [{"name": "스시명가", "rating": 4.7}, {"name": "라멘도쿄", "rating": 4.2}],
        ("Busan", "한식"): [{"name": "해운대횟집", "rating": 4.6}, {"name": "자갈치시장", "rating": 4.4}],
    }
    result = restaurants.get((city, cuisine), [{"name": "검색 결과 없음", "rating": 0}])
    return {"city": city, "cuisine": cuisine, "restaurants": result}


# 도구 실행 매핑
TOOL_FUNCTIONS = {
    "get_weather": get_weather,
    "search_restaurant": search_restaurant,
}

print("도구 함수 정의 완료")
for name, func in TOOL_FUNCTIONS.items():
    print(f"  - {name}: {func.__doc__}")

In [ ]:
# OpenAI Function Calling 용 도구 정의 (tools 파라미터)

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "지정된 도시의 현재 날씨 정보를 조회합니다. 온도, 날씨 상태, 습도 정보를 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "날씨를 조회할 도시 이름 (예: Seoul, Busan, Jeju)"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "온도 단위. 기본값: celsius"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_restaurant",
            "description": "지정된 도시에서 특정 요리 종류의 레스토랑을 검색합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "검색할 도시 이름"
                    },
                    "cuisine": {
                        "type": "string",
                        "description": "요리 종류 (예: 한식, 일식, 중식). 기본값: 한식"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

print(f"총 {len(tools)}개의 도구가 정의되었습니다.")
for t in tools:
    fn = t["function"]
    print(f"  [{fn['name']}] {fn['description']}")

In [ ]:
from openai import OpenAI

client = OpenAI()  # OPENAI_API_KEY 환경 변수 사용 (.env 에서 load_dotenv 로 로드됨)

MODEL = "gpt-4o-mini"


def run_agent(user_message: str, max_iterations: int = 5) -> str:
    """
    간단한 Agent 루프를 실행합니다.
    - 사용자 메시지를 받아 OpenAI 모델에 전달
    - tool_calls 응답이 오면 해당 도구를 실행하고 결과를 다시 전달
    - 최종 텍스트 응답이 올 때까지 반복
    """
    print(f"\n{'='*60}")
    print(f"사용자: {user_message}")
    print(f"{'='*60}")

    messages = [
        {"role": "system",
         "content": "당신은 날씨와 맛집 정보를 알려주는 친절한 어시스턴트입니다. 도구를 활용하여 정확한 정보를 제공하세요."},
        {"role": "user", "content": user_message},
    ]

    for i in range(max_iterations):
        # OpenAI API 호출
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )
        msg = response.choices[0].message

        # 도구 호출이 없으면 최종 응답
        if not msg.tool_calls:
            print(f"\nAgent 응답: {msg.content}")
            return msg.content

        # 도구 호출 처리 — assistant 메시지(도구 호출 포함)를 먼저 대화에 추가
        messages.append(msg)
        for call in msg.tool_calls:
            tool_name = call.function.name
            tool_input = json.loads(call.function.arguments or "{}")

            print(f"\n[도구 호출] {tool_name}({json.dumps(tool_input, ensure_ascii=False)})")

            # 도구 실행
            if tool_name in TOOL_FUNCTIONS:
                result = TOOL_FUNCTIONS[tool_name](**tool_input)
                print(f"[도구 결과] {json.dumps(result, ensure_ascii=False)}")
            else:
                result = {"error": f"Unknown tool: {tool_name}"}

            # 도구 결과를 role="tool" 로 추가 (tool_call_id 로 호출과 연결)
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result, ensure_ascii=False),
            })

    return "최대 반복 횟수에 도달했습니다."

In [ ]:
# Agent 실행 테스트 1: 날씨 조회
result = run_agent("서울 날씨가 어때?")

In [ ]:
# Agent 실행 테스트 2: 복합 질문 (여러 도구 호출 가능)
result = run_agent("부산 날씨 알려주고, 거기서 맛있는 한식 맛집도 추천해줘")

### Agent 루프 구조 정리

```python
while True:
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    msg = response.choices[0].message

    if not msg.tool_calls:
        # 최종 응답 → 루프 종료
        break

    # 1. 도구 호출 정보 추출 (msg.tool_calls)
    # 2. 실제 도구 실행
    # 3. 결과를 role="tool" 로 messages 에 추가
    # 4. 다음 반복으로 → 모델이 결과 기반 추가 판단
```

> **핵심**: Agent는 모델이 `tool_calls`를 요청할 때마다 도구를 실행하고,
> 그 결과를 다시 모델에게 전달하는 **루프**로 동작합니다.

## 6️⃣ MCP (Model Context Protocol) 소개

### MCP란?

**MCP (Model Context Protocol)**는 Anthropic이 제안한 **오픈 표준 프로토콜**로,
LLM 애플리케이션과 외부 데이터 소스/도구를 연결하는 표준 방식을 정의합니다.

### USB-C 비유

```
USB-C가 다양한 기기를 하나의 포트로 연결하듯,
MCP는 다양한 도구/데이터를 하나의 프로토콜로 AI에 연결합니다.

┌──────────┐     MCP      ┌──────────────┐
│  Claude  │◄────────────►│  MCP Server  │
│  (Host)  │  (Protocol)  │  (도구 제공)   │
└──────────┘              └──────┬───────┘
                                 │
                    ┌────────────┼────────────┐
                    │            │            │
               ┌────▼───┐  ┌────▼───┐  ┌────▼───┐
               │   DB   │  │  API   │  │ Files  │
               └────────┘  └────────┘  └────────┘
```

### MCP의 핵심 구성 요소

| 구성 요소 | 역할 | 설명 |
|-----------|------|------|
| **Host** | AI 애플리케이션 | Claude Desktop, IDE 등 MCP 클라이언트를 포함하는 프로그램 |
| **Client** | 프로토콜 클라이언트 | Host 내부에서 Server와 1:1 연결 유지 |
| **Server** | 도구/리소스 제공 | 외부 서비스를 MCP 프로토콜로 노출 |

### MCP Server가 제공하는 것

1. **Tools**: 외부 시스템에서 작업 수행 (API 호출, DB 쿼리 등)
2. **Resources**: 데이터 및 컨텐츠 제공 (파일 읽기, DB 데이터 등)
3. **Prompts**: 재사용 가능한 프롬프트 템플릿

### MCP Server 구성 예시

아래는 간단한 MCP Server를 Python으로 구성하는 예시입니다.

```python
# mcp_weather_server.py
from mcp.server.fastmcp import FastMCP

# MCP 서버 생성
mcp = FastMCP("Weather Service")

@mcp.tool()
def get_weather(city: str) -> str:
    """지정된 도시의 현재 날씨를 조회합니다."""
    # 실제로는 날씨 API 호출
    return f"{city}의 현재 날씨: 맑음, 18°C"

@mcp.tool()
def get_forecast(city: str, days: int = 3) -> str:
    """지정된 도시의 날씨 예보를 조회합니다."""
    return f"{city}의 {days}일간 예보: 대체로 맑음"

@mcp.resource("weather://current/{city}")
def weather_resource(city: str) -> str:
    """날씨 데이터를 리소스로 제공합니다."""
    return f"현재 {city} 날씨 데이터..."
```

### 서버 실행

```bash
# MCP 서버 실행 (stdio 전송 방식)
python mcp_weather_server.py

# 또는 SSE 전송 방식
mcp run mcp_weather_server.py --transport sse --port 8080
```

### Claude Code에서 MCP 서버 연결

Claude Code에서 MCP 서버를 연결하려면 설정 파일에 서버 정보를 추가합니다.

```bash
# Claude Code에서 MCP 서버 추가
claude mcp add weather-service python mcp_weather_server.py
```

설정 파일(`~/.claude.json`) 예시:

```json
{
  "mcpServers": {
    "weather-service": {
      "command": "python",
      "args": ["mcp_weather_server.py"],
      "type": "stdio"
    }
  }
}
```

연결 후 Claude Code에서 자연어로 날씨 도구를 사용할 수 있습니다:

```
$ claude
> 서울 날씨 알려줘
# → MCP 서버의 get_weather 도구가 자동으로 호출됨
```

### 정리: Agent 기술 스택

```
┌─────────────────────────────────────────┐
│          사용자 인터페이스                  │
│   (Claude Code, Cursor, 웹 앱 등)        │
├─────────────────────────────────────────┤
│          Agent Framework                │
│   (Agent 루프, Planning, Memory)         │
├─────────────────────────────────────────┤
│          LLM (Claude API)               │
│   (추론, Tool Use 결정)                   │
├─────────────────────────────────────────┤
│          MCP (Model Context Protocol)   │
│   (표준화된 도구/리소스 연결)               │
├─────────────────────────────────────────┤
│          External Services              │
│   (DB, API, 파일 시스템, 웹 등)           │
└─────────────────────────────────────────┘
```

**핵심 포인트:**
- **Claude Code** 자체가 Agent의 좋은 예시 (LLM + 파일시스템/터미널 도구 + 대화 메모리 + 계획 수립)
- **Function Calling**은 Agent가 외부 세계와 상호작용하는 핵심 메커니즘
- **MCP**는 도구 연결을 표준화하여 생태계를 확장하는 프로토콜

---

**© Copyright AIDENTIFY. All rights reserved.**